<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c41-header.png" alt="Nextia Learning · Speech AI: Transcribe, Understand and Speak" width="100%">

# Solution: Accents, noise and two languages

**[Accents, noise and two languages](https://learning.nextia-ai.com/courses/speech/m04/accents-noise-and-two-languages/)** · Speech AI: Transcribe, Understand and Speak · Module 4, lesson 5 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score recognisers by language, line, noise and accent, find a slice whose number is really about another slice, check Whisper's language detection, and listen to a French caller who switches to English.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (a few MB). No account, no key, and no model download: this notebook reads recorded transcripts. |
| **You should know** | WER after the normalisation guide and test sets by slice, from [The errors that matter](https://learning.nextia-ai.com/courses/speech/m02/the-errors-that-matter/); the Whisper sizes, from [Whisper sizes on a laptop](https://learning.nextia-ai.com/courses/speech/m04/whisper-sizes-on-a-laptop/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, SciPy 1.18.1, soundfile 0.14.0 and matplotlib 3.11.2 (scikit-learn 1.9.1, jiwer 4.0.0 and faster-whisper 1.2.1 where the lesson needs them), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/speech/m04/accents-noise-and-two-languages/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/M04-L05-accents-noise-and-two-languages/accents-noise-and-two-languages-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, scipy 1.18.1, soundfile 0.14.0, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 scipy==1.18.1 soundfile==0.14.0 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'scipy', 'soundfile']:
    print(name, version(name))

## Setup: get the Larkfield calls

Run the next cell. It downloads the files that this lesson needs (about 4 MB) into a folder `larkfield-calls`, with the course's helper `compare.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-calls` (in Colab: the **Files** panel on the left), then run the cell again.

Where the calls come from: Larkfield is an invented shop. Our code wrote the scripts, and open text-to-speech voices (Piper and Kokoro) read them. Every call is **synthetic**. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C41/data/dataset-card.md) names each voice and its licence. The calls are free to reuse under CC BY 4.0.

In [ ]:
# Setup: download the files into larkfield-calls/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C41/"
FILES = {
    "data/calls/calls.jsonl": "0ab1c11490f1e0341f11f9cf0fc061abde616a9297fbc830cde001ef60daa01a",
    "data/calls/phone/C-0009.wav": "e69f30d30f0e1b73dedbe6a13304dfc45d749112339b35939171ee1a8223e2e2",
    "data/recordings/asr/vosk-small.jsonl": "ad050936f74c20db430f7668e4baa5e97ee84ce9cfdf5f0f8d98e8a8f024fe4d",
    "data/recordings/asr/whisper-tiny.jsonl": "6c6026f6b4971dd04e945beeb3d4a155eacc6ba54d77452dff27f5082c9f3104",
    "data/recordings/asr/whisper-base.jsonl": "206b63ff863dd96fcbadf8034a8001e902ce06335cadf5348631b1a8d5d90420",
    "data/recordings/asr/whisper-small.jsonl": "67f27c0e6f5aa73d0e67cdb4b69c0e2d22c5f03a5dd74be14cbd8db0df17cb3f",
    "data/recordings/asr/whisper-small-auto.jsonl": "4e2f21b8c0667306305e75fcaee792080697e87cc8314e71d0815164d0f619eb",
    "data/recordings/asr/whisper-small-prompt.jsonl": "3089427ee976a04c38d6d4d2f505e22acd48028cdc1d6560d5bdcae8b364be2f",
    "compare.py": "db394eb044e0423706644ba7c8e6451e4d21a268293b868032918ea2f30db85b",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-calls").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-calls and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-calls.")

## Setup: helpers

The next cell loads NumPy as `np`, soundfile as `sf`, SciPy's `signal`, matplotlib, the audio player `Audio`, and the course helper as `cp` (normalisation, word error rate and the comparison table). It defines a few small helpers:

- `load_calls()` reads the labels of every call; `reference(call)` joins its turns into the reference transcript.
- `read_audio(path, sr, channel)` reads a recording as numbers between -1 and 1; `dbfs(x)` gives its loudness.
- `true_turns(call)` gives the true (start, end, speaker) of each turn, from the first word to the last word.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: calls, audio, and checks that do not show the answer.
import hashlib, json, warnings
from math import gcd
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
import compare as cp
warnings.filterwarnings("ignore")

def load_calls(path="data/calls/calls.jsonl"):
    """The labels of every call, by call ID."""
    rows = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]
    return {c["call_id"]: c for c in rows}

def reference(call, speaker=None):
    """The reference transcript: the text of all turns (or of one speaker's turns), in time order."""
    return " ".join(t["text"] for t in call["turns"] if speaker in (None, t["speaker"]))

def read_audio(path, sr=None, channel=None):
    """(numbers between -1 and 1, samples a second). channel=None mixes the channels; 0 or 1 keeps one side."""
    x, file_sr = sf.read(str(path), dtype="float32", always_2d=True)
    x = x.mean(axis=1) if channel is None else x[:, channel]
    if sr and sr != file_sr:
        g = gcd(file_sr, sr)
        x = signal.resample_poly(x, sr // g, file_sr // g).astype(np.float32)
    return x, sr or file_sr

def dbfs(x):
    """Loudness in decibels below full scale: 0 is the loudest; speech on a phone is often -30 to -15."""
    return float(20 * np.log10(np.sqrt(np.mean(np.square(x))) + 1e-9))

def true_turns(call):
    """The true turns: (start, end, speaker), from the first word to the last word of each turn."""
    return [(t["words"][0]["start"], t["words"][-1]["end"], t["speaker"]) if t.get("words")
            else (t["start"], t["end"], t["speaker"]) for t in call["turns"]]

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The recorded transcripts

Every call is synthetic: open text-to-speech voices read our scripts. The English calls use many voices; the French calls use only **three** voices, and part of the French error comes from those voices themselves. Run the next cell to load the labels and the recorded transcripts of four recognisers, and two variants of Whisper small: one that detected the language itself (`whisper-small-auto`) and one with Larkfield's vocabulary prompt.

In [ ]:
calls = load_calls()
test = [c for c in calls.values() if c["split"] == "test"]
methods = ['vosk-small', 'whisper-tiny', 'whisper-base', 'whisper-small', 'whisper-small-auto', 'whisper-small-prompt']
recorded = {}   # recorded[method][call_id] = the row of the phone transcript
for m in methods:
    rows = [json.loads(line) for line in open(f"data/recordings/asr/{m}.jsonl", encoding="utf-8") if line.strip()]
    recorded[m] = {r["call_id"]: r for r in rows if r["kind"] == "phone"}
print(len(test), "test calls")

## 2. WER by language

> **Predict.** Which recogniser loses the most when it moves from English to French?

In [ ]:
def wer(method, items):
    """Corpus WER after the guide, in %, for a list of calls."""
    s = cp.corpus_scores([(reference(c), recorded[method][c["call_id"]]["text"], c["language"]) for c in items])
    return round(100 * s["wer"], 1)

def by(method, column, items=None):
    """WER for each value of one label column (language, line, noise, accent), with the number of calls."""
    groups = {}
    for c in items or test:
        groups.setdefault(c[column], []).append(c)
    return {k: (len(v), wer(method, v)) for k, v in sorted(groups.items())}

for m in ["vosk-small", "whisper-tiny", "whisper-base", "whisper-small"]:
    print(f"{m:14}", by(m, "language"))

> **Check.** Vosk small: English 24.3, French 39.4. Whisper tiny: 7.0 and 37.0. Whisper small: 3.1 and 15.1. On these calls, the language is the biggest slice difference of all.

## 3. WER by line

Larkfield's calls come on three lines: landline (G.711), mobile (Opus at 8 kbit/s) and internet phone (VoIP, G.726 at 16 kbit/s).

In [ ]:
print("Whisper small by line, all calls:", by("whisper-small", "line"))

> **Check.** Landline 8.6, mobile 5.9, VoIP 12.2. It looks as if the VoIP codec doubles the errors. Before you believe it, look at **who** is on each line.

In [ ]:
from collections import Counter
print(Counter((c["line"], c["language"]) for c in test))
for language in ("en", "fr"):
    print(language, by("whisper-small", "line", [c for c in test if c["language"] == language]))

> **Check.** The mobile calls are mostly English (8 of 10); the VoIP calls are mostly French (4 of 7). Inside one language, the lines are close: English landline 3.1, mobile 3.0, VoIP 3.5. The "line effect" was mostly a **language effect**. With 3 to 10 calls in a cell, small differences are noise.

## 4. Your turn: noise, inside one language

All calls: quiet 9.6, street 7.4. Quiet calls look harder than street calls. That is odd.

> **Your turn.** Compute Whisper small's WER by `noise` on the **English calls only**. Put the result in a dictionary `noise_english` (noise → WER in %, as `by` gives it, without the call counts).

In [ ]:
english = [c for c in test if c["language"] == "en"]
noise_english = {k: w for k, (n, w) in by("whisper-small", "noise", english).items()}
print("all calls:   ", by("whisper-small", "noise"))
print("English only:", noise_english)

In [ ]:
expect_hash('noise_english', noise_english, '83eb470b7c923ae5')

> **Check.** English only: quiet 2.3, background talk 3.3, street 4.3. Inside English, noise does hurt a little, as you would expect. Half of the quiet calls are French, which is why "quiet" looked worse in the overall table.

## 5. Should Whisper detect the language?

Whisper can listen to the start of a call and guess the language. One run let it do so.

In [ ]:
from collections import Counter
found = Counter((c["language"], recorded["whisper-small-auto"][c["call_id"]]["language_detected"]) for c in test)
print("(true language, detected language):", dict(found))
print("WER told the language:", wer("whisper-small", test), "  detected:", wer("whisper-small-auto", test))

> **Check.** It found the right language on every test call, and the WER is almost the same (8.7 against 8.6 when told). On Larkfield's calls, each call stays in one language. A call that changes language is harder: next.

## 6. A French caller who switches to English

Call C-0009: the caller speaks French, uses the English product name "Cedar raised bed", then says one sentence in English. Listen, and read what Whisper small wrote when it was told "French".

In [ ]:
c = calls["C-0009"]
for t in c["turns"][1:3]:
    print(f"{t['speaker']:6} {t['start']:5.1f} s  {t['text']}")
x, sr = read_audio("data/calls/phone/C-0009.wav", channel=1)      # the caller's channel
display(Audio(x[int(8.5 * sr):int(20.3 * sr)], rate=sr))
text = recorded["whisper-small"]["C-0009"]["text"]
print("\nWhisper small, told 'French':", text[text.find("Oui"):text.find("Je peux")])

> **Check.** "le SEDAR, le BED" for "le Cedar raised bed", and "Soil, en anglais. The appseid de D.V. is offline." for "Sorry, in English: the app says the device is offline." Told "French", the model writes French-looking words for English speech. Larkfield's French callers say the English product names in almost every call.

In [ ]:
for m in ["whisper-small", "whisper-small-prompt"]:
    for language in ("en", "fr"):
        rows = [cp.important_errors(c["important"], recorded[m][c["call_id"]]["text"], c["language"])
                for c in test if c["language"] == language]
        p = cp.sum_important(rows)["product"]
        print(f"{m:21} {language}: product names missed {p['missed']} of {p['items']}")

> **Check.** Without the prompt, Whisper small misses 25 of the 26 product names in French calls. With the vocabulary prompt, it misses 7. For a bilingual shop, the vocabulary helps most where the languages mix.

## Next

You scored by language, line and noise, found two slices that were really about the language, and heard what a language switch does. The [lesson page](https://learning.nextia-ai.com/courses/speech/m04/accents-noise-and-two-languages/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Split by channel first](https://learning.nextia-ai.com/courses/speech/m05/split-by-channel-first/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/speech/m04/accents-noise-and-two-languages/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). Every call is synthetic: open text-to-speech voices read scripts that we invented. No real customer and no real person's voice is in this data.